In [1]:
from pathlib import Path
import re
import pandas as pd

In [2]:
ROOT = Path.cwd().parent

RUNTIME_DIR = ROOT / "dataset" / "runtime"

INPUT_PATH = RUNTIME_DIR / "comments_raw.csv"
OUTPUT_PATH = RUNTIME_DIR / "comments_aspects.csv"

RUNTIME_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Input comments:", INPUT_PATH)
print("Aspect-level output:", OUTPUT_PATH)

Project root: c:\New folder\Projects\sentiment analysis
Input comments: c:\New folder\Projects\sentiment analysis\dataset\runtime\comments_raw.csv
Aspect-level output: c:\New folder\Projects\sentiment analysis\dataset\runtime\comments_aspects.csv


#### Temporary

In [3]:
if not INPUT_PATH.exists():
    sample_comments = pd.DataFrame(
        [
            {
                "comment_id": "c001",
                "text": (
                    "The laptop is excellent for school work and browsing, "
                    "but 8GB RAM is restrictive."
                )
            },
            {
                "comment_id": "c002",
                "text": (
                    "Light games and older titles run fine, "
                    "but AAA gaming will struggle."
                )
            },
            {
                "comment_id": "c003",
                "text": (
                    "The touchscreen is useful, and Microsoft Office "
                    "works smoothly."
                )
            },
            {
                "comment_id": "c004",
                "text": (
                    "The laptop has poor gaming performance, although "
                    "it is fine for normal productivity."
                )
            },
            {
                "comment_id": "c005",
                "text": (
                    "The battery life is decent, but storage fills up "
                    "quickly after installing games."
                )
            }
        ]
    )

    sample_comments.to_csv(INPUT_PATH, index=False)

    print("Created sample comments file:")
    print(INPUT_PATH)
else:
    print("comments_raw.csv already exists; using existing file.")

Created sample comments file:
c:\New folder\Projects\sentiment analysis\dataset\runtime\comments_raw.csv


In [4]:
comments = pd.read_csv(INPUT_PATH)

print("Original comment count:", len(comments))
print("Columns:", comments.columns.tolist())

if "comment_id" not in comments.columns:
    comments["comment_id"] = [
        f"comment_{index:05d}"
        for index in range(len(comments))
    ]

if "text" not in comments.columns:
    raise ValueError(
        "comments_raw.csv must contain a 'text' column."
    )

comments = comments.dropna(subset=["text"]).copy()

comments["text"] = comments["text"].astype(str).str.strip()

comments = comments[
    comments["text"].ne("")
].copy()

# Remove exact duplicate comment text.
comments = comments.drop_duplicates(
    subset=["text"]
).reset_index(drop=True)

# Optional: remove extremely short/noisy comments.
comments = comments[
    comments["text"].str.len() >= 10
].reset_index(drop=True)

print("Clean comment count:", len(comments))

display(comments.head())

Original comment count: 5
Columns: ['comment_id', 'text']
Clean comment count: 5


,comment_id,text
0,c001,The laptop is excellent for school work and br...
1,c002,"Light games and older titles run fine, but AAA..."
2,c003,"The touchscreen is useful, and Microsoft Offic..."
3,c004,"The laptop has poor gaming performance, althou..."
4,c005,"The battery life is decent, but storage fills ..."


In [5]:
ASPECT_PATTERNS = {
    "8gb ram": [
        r"\b8\s*gb\s*(of\s*)?(ram|memory)\b",
        r"\b8gb\s*(ram|memory)?\b",
        r"\b8\s*gigs?\b",
        r"\b8\s*gb\b",
    ],

    "ram": [
        r"\bram\b",
        r"\bmemory\b",
        r"\bsystem\s+memory\b",
    ],

    "gaming performance": [
        r"\bgaming\s+performance\b",
        r"\bserious\s+gaming\b",
        r"\bmodern\s+gaming\b",
        r"\btriple\s*a\b",
        r"\baaa\b",
        r"\bcompetitive\s+gaming\b",
        r"\besports?\b",
        r"\bgraphically\s+intensive\b",
        r"\bhigh[-\s]?end\s+games?\b",
        r"\bgame(s)?\s+(will|won't|will\s+not|cannot|can't)\s+(run|struggle)\b",
        r"\bnot\s+(a\s+)?gaming\s+(laptop|computer)\b",
    ],

    "light gaming": [
        r"\blight\s+gaming\b",
        r"\bcasual\s+gaming\b",
        r"\bcasual\s+games?\b",
        r"\bold(er)?\s+games?\b",
        r"\bpre[-\s]?2020\s+games?\b",
        r"\bindie\s+games?\b",
        r"\bcloud\s+gaming\b",
        r"\bgame\s*pass\b",
        r"\bgeforce\s+now\b",
        r"\bbrowser[-\s]?based\s+games?\b",
        r"\broblox\b",
        r"\bminecraft\b",
        r"\bterraria\b",
        r"\bretro\s+games?\b",
    ],

    "school": [
        r"\bschool\b",
        r"\bschool\s+work\b",
        r"\bcollege\b",
        r"\buniversity\b",
        r"\bstudent\b",
        r"\bhomework\b",
        r"\bclass(es)?\b",
    ],

    "study": [
        r"\bstud(y|ying|ies)\b",
        r"\bnotes?\b",
        r"\bassignment(s)?\b",
        r"\blearning\b",
    ],

    "office work": [
        r"\boffice\s+work\b",
        r"\boffice\b",
        r"\bwork\s+laptop\b",
        r"\bwork\s+computer\b",
        r"\bdocuments?\b",
    ],

    "productivity": [
        r"\bproductivity\b",
        r"\bmicrosoft\s+office\b",
        r"\bmicrosoft\s+word\b",
        r"\bms\s+word\b",
        r"\bexcel\b",
        r"\bpowerpoint\b",
        r"\bword\b",
        r"\bprogramming\b",
        r"\bcoding\b",
        r"\btext\s+editing\b",
        r"\bweb\s+browsing\b",
        r"\bbrowsing\b",
        r"\bdiscord\b",
    ],

    "touchscreen": [
        r"\btouchscreen\b",
        r"\btouch\s+screen\b",
        r"\btouch[-\s]?enabled\b",
        r"\btouch\s+display\b",
    ],

    "general performance": [
        r"\bgeneral\s+performance\b",
        r"\bgeneral\s+use\b",
        r"\bgeneral\s+usage\b",
        r"\beveryday\s+(task|tasks|use|usage)\b",
        r"\bdaily\s+(task|tasks|use|usage)\b",
        r"\bbasic\s+(task|tasks|use|usage)\b",
        r"\bnormal\s+(task|tasks|use|usage)\b",
        r"\bmost\s+tasks?\b",
        r"\bfor\s+basic\s+stuff\b",
    ],

    "storage": [
        r"\bstorage\b",
        r"\bssd\b",
        r"\b512\s*gb\b",
        r"\b512gb\b",
        r"\bdisk\s+space\b",
        r"\bhard\s+drive\b",
    ],

    "battery": [
        r"\bbattery\b",
        r"\bbattery\s+life\b",
        r"\bpower\s+bank\b",
        r"\bcharging\b",
        r"\bcharger\b",
    ],

    "screen": [
        r"\bscreen\b",
        r"\bdisplay\b",
        r"\bbrightness\b",
        r"\bresolution\b",
        r"\b1080p\b",
        r"\b1200p\b",
        r"\b2k\b",
        r"\bips\b",
    ],

    "cpu performance": [
        r"\bcpu\b",
        r"\bprocessor\b",
        r"\bcore\s+ultra\b",
        r"\bcore\s+[3579]\b",
        r"\bi[3579]\b",
    ],

    "graphics performance": [
        r"\bigpu\b",
        r"\bintegrated\s+graphics\b",
        r"\bintegrated\s+gpu\b",
        r"\bdedicated\s+gpu\b",
        r"\bgraphics\s+card\b",
        r"\bgpu\b",
        r"\bvram\b",
    ],

    "upgradeability": [
        r"\bupgradeable\b",
        r"\bupgrade\s+(ram|memory|storage|ssd)\b",
        r"\bnot\s+soldered\b",
        r"\bmemory\s+slot\b",
        r"\bram\s+slot\b",
    ],

    "build quality": [
        r"\bbuild\s+quality\b",
        r"\bcheap\s+(frame|body|plastic)\b",
        r"\bframe\b",
        r"\bbody\b",
        r"\bchassis\b",
        r"\bplastic\b",
    ],

    "hinge quality": [
        r"\bhinge(s)?\b",
    ],

    "value for money": [
        r"\bvalue\s+for\s+money\b",
        r"\bgood\s+value\b",
        r"\bworth\s+(it|the\s+price)\b",
        r"\boverpriced\b",
        r"\bcheap\s+for\s+the\s+price\b",
        r"\bprice\b",
        r"\bbudget\b",
        r"\bdeal\b",
    ],

    "ai performance": [
        r"\bai\b",
        r"\bcopilot\b",
        r"\bnpu\b",
        r"\blocal\s+models?\b",
        r"\bllm\b",
        r"\b7b\b",
    ],
}

In [6]:
def extract_matched_phrase(text, pattern):
    """
    Return the first exact phrase in the comment that matches a regex.
    """
    match = re.search(
        pattern,
        text,
        flags=re.IGNORECASE
    )

    if match:
        return match.group(0)

    return None

In [7]:
def extract_aspects(text):
    """
    Returns a list of dictionaries:
    [
        {
            'raw_aspect': '8GB RAM',
            'aspect_term': '8gb ram'
        },
        ...
    ]
    """
    text = str(text).lower()

    extracted = []

    for normalized_aspect, patterns in ASPECT_PATTERNS.items():
        for pattern in patterns:
            matched_phrase = extract_matched_phrase(
                text,
                pattern
            )

            if matched_phrase:
                extracted.append(
                    {
                        "raw_aspect": matched_phrase,
                        "aspect_term": normalized_aspect
                    }
                )
                break

    # Avoid duplicate generic RAM if an 8GB-RAM-specific phrase exists.
    found_aspects = {
        item["aspect_term"]
        for item in extracted
    }

    if "8gb ram" in found_aspects and "ram" in found_aspects:
        extracted = [
            item
            for item in extracted
            if item["aspect_term"] != "ram"
        ]

    # Avoid duplicate generic gaming verdict if the comment only says light gaming.
    # Keep both only if the text includes a clear heavy/AAA/serious gaming phrase.
    light_game_terms = {
        "light gaming"
    }

    if (
        "light gaming" in found_aspects
        and "gaming performance" in found_aspects
    ):
        strong_gaming_patterns = [
            r"\baaa\b",
            r"\btriple\s*a\b",
            r"\bcompetitive\b",
            r"\besports?\b",
            r"\bnot\s+(a\s+)?gaming\b",
            r"\bstruggle\b",
            r"\bwon't\s+run\b",
            r"\bwill\s+not\s+run\b",
        ]

        strong_gaming_mentioned = any(
            re.search(pattern, text)
            for pattern in strong_gaming_patterns
        )

        if not strong_gaming_mentioned:
            extracted = [
                item
                for item in extracted
                if item["aspect_term"] != "gaming performance"
            ]

    return extracted

In [ ]:
test_comment = """
The laptop is great for school work, browsing, coding,
and Microsoft Office. However, 8GB RAM is restrictive.
Light games and older games work, but AAA gaming will struggle.
The touchscreen is useful.
"""

extracted_test_aspects = extract_aspects(test_comment)

pd.DataFrame(extracted_test_aspects)

,raw_aspect,aspect_term
0,8gb ram,8gb ram
1,aaa,gaming performance
2,older games,light gaming
3,school,school
4,office,office work
5,microsoft office,productivity
6,touchscreen,touchscreen


In [9]:
aspect_rows = []

for _, row in comments.iterrows():
    comment_id = row["comment_id"]
    text = row["text"]

    extracted_aspects = extract_aspects(text)

    for aspect_info in extracted_aspects:
        aspect_rows.append(
            {
                "comment_id": comment_id,
                "text": text,
                "raw_aspect": aspect_info["raw_aspect"],
                "aspect_term": aspect_info["aspect_term"]
            }
        )

comments_aspects = pd.DataFrame(aspect_rows)

print("Comments processed:", len(comments))
print("Comment-aspect rows created:", len(comments_aspects))

display(comments_aspects.head(30))

Comments processed: 5
Comment-aspect rows created: 11


,comment_id,text,raw_aspect,aspect_term
0,c001,The laptop is excellent for school work and br...,8gb ram,8gb ram
1,c001,The laptop is excellent for school work and br...,school,school
2,c001,The laptop is excellent for school work and br...,browsing,productivity
3,c002,"Light games and older titles run fine, but AAA...",aaa,gaming performance
4,c003,"The touchscreen is useful, and Microsoft Offic...",office,office work
5,c003,"The touchscreen is useful, and Microsoft Offic...",microsoft office,productivity
6,c003,"The touchscreen is useful, and Microsoft Offic...",touchscreen,touchscreen
7,c004,"The laptop has poor gaming performance, althou...",gaming performance,gaming performance
8,c004,"The laptop has poor gaming performance, althou...",productivity,productivity
9,c005,"The battery life is decent, but storage fills ...",storage,storage


In [10]:
if len(comments_aspects) == 0:
    print(
        "No aspects were detected. Review ASPECT_PATTERNS "
        "or inspect comments_raw.csv."
    )
else:
    aspect_counts = (
        comments_aspects["aspect_term"]
        .value_counts()
        .rename_axis("aspect_term")
        .reset_index(name="mentions")
    )

    display(aspect_counts)

,aspect_term,mentions
0,productivity,3
1,gaming performance,2
2,school,1
3,8gb ram,1
4,office work,1
5,touchscreen,1
6,storage,1
7,battery,1


In [11]:
comments_aspects.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Saved aspect-level data to:")
print(OUTPUT_PATH)

print("\nOutput columns:")
print(comments_aspects.columns.tolist())

Saved aspect-level data to:
c:\New folder\Projects\sentiment analysis\dataset\runtime\comments_aspects.csv

Output columns:
['comment_id', 'text', 'raw_aspect', 'aspect_term']


In [12]:
if len(comments_aspects) > 0:
    model_input_preview = comments_aspects.copy()

    model_input_preview["tfidf_input"] = (
        model_input_preview["aspect_term"]
        + " [SEP] "
        + model_input_preview["text"]
    )

    display(
        model_input_preview[
            [
                "comment_id",
                "raw_aspect",
                "aspect_term",
                "tfidf_input"
            ]
        ].head(15)
    )

,comment_id,raw_aspect,aspect_term,tfidf_input
0,c001,8gb ram,8gb ram,8gb ram [SEP] The laptop is excellent for scho...
1,c001,school,school,school [SEP] The laptop is excellent for schoo...
2,c001,browsing,productivity,productivity [SEP] The laptop is excellent for...
3,c002,aaa,gaming performance,gaming performance [SEP] Light games and older...
4,c003,office,office work,"office work [SEP] The touchscreen is useful, a..."
5,c003,microsoft office,productivity,"productivity [SEP] The touchscreen is useful, ..."
6,c003,touchscreen,touchscreen,"touchscreen [SEP] The touchscreen is useful, a..."
7,c004,gaming performance,gaming performance,gaming performance [SEP] The laptop has poor g...
8,c004,productivity,productivity,productivity [SEP] The laptop has poor gaming ...
9,c005,storage,storage,"storage [SEP] The battery life is decent, but ..."
